# Dependency Caching with uv — Bob Follows the Cache

*A project, two versions of uv, and a cache Bob can inspect before he prunes it.*

## Introduction — Bob’s Cache Question

> **Bob:** Yesterday I built a wheel, today I downloaded another. If I prune the cache, which one will I need to fetch or build again? I want to see the paths disappear, not just take the command's word for it.

Run this notebook top to bottom in the [uv Cache Lab Dev Container](./README.md) at `/workspace`. The image supplies uv 0.13.3, Python 3.12, and the Jupyter kernel; the cells populate the package cache. The chapter [Dependency Caching with uv](../../docs/chapter-03/section-03/uv_cache.md) explains the same examples. A first run needs network access. All commands below execute **inside the container**; `lab/` is ignored by Git and the container-local uv cache is separate from your host's cache.

The notebook follows the chapter's order. Its tabs become subchapters, and each cleanup stage has its own *restore → measure → clean up → inspect* sequence. Cache bucket suffixes are specific to the pinned uv versions, not a public storage API.

## Applied Lab

> **Bob:** I'll keep a working project environment and a synthetic 16 MiB fixture. Then I can compare what is installed with what uv keeps in its cache.

The locked demo uses NumPy, SciPy, and Matplotlib. Its `uv-cache-demo` wheel bundles deterministic generated data; it is **not** a trained model. This lab sets `UV_LINK_MODE=copy`, so installed packages in the main `.venv` do not depend on cached files surviving cleanup. The old-version experiment uses `uvx` to run uv 0.4.12, but reserves the actual pruning for uv 0.13.3.

## Building Blocks

### Environment Variables That Influence Cache Behavior

> **Bob:** Before I delete anything, which knob chooses the cache and which knob chooses how files get into my environment?

| Variable | What it controls |
| --- | --- |
| `UV_CACHE_DIR` | Package-cache location; `--cache-dir` overrides it for the invoked uv. |
| `UV_NO_CACHE` | A disposable invocation-specific cache. |
| `UV_OFFLINE` | No network; installation needs reusable cached or local artifacts. |
| `UV_LINK_MODE` | `clone`, `copy`, `hardlink`, or `symlink` when installing from cache. |
| `UV_LOCK_TIMEOUT` | Time uv waits for other cache users during cleanup. |
| `UV_CONCURRENT_CACHE_READS` | Threads for cached HTTP response reads. |

Use `uv cache dir` to identify the directory you are about to examine. Copy mode is the lab's default; the link-mode experiments temporarily override it.

### Commands That Populate the Cache

`uv pip install`, `uv add`, `uv sync`, and `uv tool install` can populate or reuse cached package artifacts. `--refresh` revalidates entries; `--offline` tests whether an install can reuse what is already there. `uv cache prune`, `uv cache prune --ci`, and `uv cache clean` remove different entries, which Bob will measure below.

### Build Configuration and Wheel Reuse

> **Bob:** If I change the data in my local wheel, a reused wheel would lie to me. I need a key that tracks the build inputs, not just the project's name.

uv normally checks local build inputs such as `pyproject.toml`. A project can set `tool.uv.cache-keys` to track extra files, Git commits, or environment variables; custom keys **replace** the defaults. This lab uses `cache-keys = [{ env = "CACHE_DEMO_KEY" }]`. After generating the fixture, `scripts/build_key.py` fingerprints the package's source, manifest, JSON, and data. Every `uv sync` that installs the local wheel must supply `CACHE_DEMO_KEY`; `--no-editable` actually builds a wheel rather than linking the source tree.

### Prepare the Container Lab

> **Bob:** I want a repeatable starting point, so I will reset only this lab's generated workspace files and its container-local package cache.

This cell clears the container's default uv cache and the ignored lab artifacts. Run it **before** the population chapters, not between their steps. The Python installation is outside the package cache.

In [ ]:
%%bash
set -euxo pipefail
pwd
uv --version
uv cache dir
rm -rf lab .venv reports src/uv_cache_demo/data/embeddings.bin
mkdir -p lab/prune-probe
uv cache clean


## Populate and Inspect the Cache

uv stores downloaded wheels, locally built wheels, unpacked source trees, and (with an opt-in feature) centralized project environments. The last target is old-format data created by uv 0.4.12. Bob checks that each exists **before** asking uv to remove it.

### Downloaded Wheels

> **Bob:** NumPy and its friends already ship wheels. Let me install them without building anything, then look at the cache they leave behind.

Create the project's Python 3.12 environment. `--only-binary :all:` requires wheels from the index and keeps this first example distinct from later source builds. The main `.venv` is outside the package cache.

In [ ]:
%%bash
set -euxo pipefail
uv venv --python 3.12
uv pip install --python .venv/bin/python --only-binary :all: numpy scipy matplotlib
uv cache size --human --preview-features cache-size
find "$(uv cache dir)" -type d \( -path '*/wheels-v*/pypi/numpy' -o -path '*/wheels-v*/pypi/scipy' -o -path '*/wheels-v*/pypi/matplotlib' \) -print


### Locally Built Wheel

> **Bob:** Downloading is cheap compared with rebuilding my own package. I'll give uv a content fingerprint and make it produce a real wheel.

Generate the deterministic 16 MiB fixture, synchronize the locked project as a non-editable install, and run the installed program. The wheel in the source-distribution cache is our later checkpoint; the report and installed package remain outside that cache.

In [ ]:
%%bash
set -euxo pipefail
.venv/bin/python scripts/generate_fixture.py
CACHE_DEMO_KEY=$(.venv/bin/python scripts/build_key.py) uv sync --locked --no-editable --python 3.12
.venv/bin/python -m uv_cache_demo.main --output reports/embeddings.png
find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print
uv cache size --human --preview-features cache-size


### Unpacked Source Distribution

> **Bob:** A source archive is more than the wheel it produces. I want to see both the extracted source tree and the resulting wheel before CI pruning separates them.

Force an `idna==3.10` source build into an ignored `--target` directory. `--no-deps` leaves the project's lockfile alone. In uv 0.13.3, the `sdists-v10` bucket contains both the built `.whl` and an unpacked `src/idna` tree. These paths describe cached build artifacts, **not** the `lab/sdist-target/idna` installation.

In [ ]:
%%bash
set -euxo pipefail
uv pip install --python "$(uv python find 3.12)" --target lab/sdist-target --no-deps --no-binary idna 'idna==3.10'
find "$(uv cache dir)" -type d -name 'sdists-v*' -print
find "$(uv cache dir)" -type d -path '*/sdists-v*/pypi/idna/3.10/*/src/idna' -print
find "$(uv cache dir)" -type f -name 'idna-*.whl' -print
uv cache size --human --preview-features cache-size


### Centralized Project Environment

> **Bob:** My main `.venv` sits beside the project. Can uv put a *different* project's environment in its cache so I can see what prune does to it?

Copy the locked declarations into an ignored project. With `centralized-project-envs`, the default `.venv` points into the cache (usually by symlink). Do not set `UV_PROJECT_ENVIRONMENT` or `--active` here: those explicit environment selections bypass centralization. Save the resolved target while the link still works.

In [ ]:
%%bash
set -euxo pipefail
mkdir -p lab/centralized
cp pyproject.toml uv.lock lab/centralized/
uv --preview-features centralized-project-envs sync --directory lab/centralized --locked --no-install-project --python 3.12
ls -ld lab/centralized/.venv
readlink -f lab/centralized/.venv > lab/centralized-target
test -d "$(<lab/centralized-target)"
lab/centralized/.venv/bin/python -c 'import numpy; print(numpy.__version__)'


### Other Unused Cache Entries — An Older uv

> **Bob:** Does ‘unused’ mean I uninstalled a package? Or does it mean this version of uv cannot use the old cache format? I'll keep the installation and change only the cache's age.

Run uv **0.4.12** through `uvx` with a **separate bootstrap cache**. The `--cache-dir "$(uv cache dir)"` argument belongs to the *old uv executable*, so it writes to the same package cache as the current uv. Old uv uses `wheels-v1` for prebuilt `packaging` and `built-wheels-v3` for both an unpacked `idna` sdist and its built wheel. Current uv uses `wheels-v7` and `sdists-v10`. Both old installs go into independent `--target` directories; pruning the cache does not inspect or delete those installed copies.

Cache bucket versions are an implementation detail. Check the actual directories before relying on these names, which are specific to the versions pinned in this lab.

In [ ]:
%%bash
set -euxo pipefail
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --version
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$(uv cache dir)" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/old-wheel --reinstall --no-deps --only-binary packaging 'packaging==24.2'
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$(uv cache dir)" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/old-sdist --reinstall --no-deps --no-binary idna 'idna==3.10'
find "$(uv cache dir)" -mindepth 1 -maxdepth 1 -type d -printf '%f\n' | sort
find "$(uv cache dir)/built-wheels-v3/pypi/idna/3.10" -type f -name 'idna-*.whl' -print
test -d lab/prune-probe/old-sdist/idna


## Reuse Cached Packages

### Offline Synchronization

> **Bob:** If I throw away `.venv` but keep the package cache, can uv rebuild my environment with the network switched off?

Remove only the ordinary project environment. The numerical wheels and the locally built wheel remain cached; the generated fixture and its fingerprint have not changed. An offline sync should restore `.venv` without a download.

In [ ]:
%%bash
set -euxo pipefail
rm -rf .venv
CACHE_DEMO_KEY=$("$(uv python find 3.12)" scripts/build_key.py) uv sync --locked --no-editable --offline --python 3.12
.venv/bin/python -c 'import numpy, uv_cache_demo; print(numpy.__file__, uv_cache_demo.__file__)'


### Compare UV_LINK_MODE

> **Bob:** Reusing the same wheel does not always mean copying the same bytes. Let me compare how uv connects the cache to four disposable environments.

These are *installation modes*, not separate caches. Each `--offline` sync uses the locked dependency wheels and installs into a different ignored environment. On a bind-mounted workspace, `clone` or `hardlink` may fall back to copying if the cache and workspace are on different filesystems. `stat` shows hard-link counts but cannot by itself prove copy-on-write cloning.

#### Clone — The Linux Default

> **Bob:** If the filesystem supports it, a clone shares blocks until someone changes a file. I'll ask for `clone` explicitly because this lab normally selects `copy`.

Inspect NumPy in the disposable environment and verify it imports:

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/clone UV_LINK_MODE=clone uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/clone/lib/python3.12/site-packages/numpy/__init__.py
lab/clone/bin/python -c 'import numpy; print(numpy.__version__)'


#### Copy — The Lab’s Default

> **Bob:** A full copy costs storage, but it won't vanish when I empty uv's cache. That is the tradeoff I chose for my main `.venv`.

Inspect the independent copy and its link count:

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/copy UV_LINK_MODE=copy uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/copy/lib/python3.12/site-packages/numpy/__init__.py


#### Hardlink — Share Files on One Filesystem

> **Bob:** A hardlink is more intimate than a copy: if two paths share an inode, editing one changes the other. Does this workspace even share a filesystem with the cache?

A count above one suggests a real hardlink; a count of one or an uv fallback warning means it probably copied across filesystem boundaries.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/hardlink UV_LINK_MODE=hardlink uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/hardlink/lib/python3.12/site-packages/numpy/__init__.py


#### Symlink — Depend on the Cache

> **Bob:** This time the installed file points *into* the cache. Once I prune, that link might dangle, so I will inspect it and discard this test environment first.

The main `.venv` is still copy-mode. Removing this symlink-mode trial before cleanup keeps the subsequent import checks meaningful.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/symlink UV_LINK_MODE=symlink uv sync --locked --no-install-project --offline --python 3.12
readlink lab/symlink/lib/python3.12/site-packages/numpy/__init__.py
rm -rf lab/symlink


## Cache Cleanup Operations

> **Bob:** The chapter shows three cleanup commands as tabs, but tabs hide the order in which I ran them. I'll put them one after another here, with a **freshly verified starting cache** for each command.

Run ordinary prune, CI prune, then complete clean. Normal prune removes old buckets and the centralized environment. Before CI prune Bob **recreates** those targets; before clean he recreates all five again. We save sorted path inventories in `lab/prune-probe/` (outside the cache) and use `comm -23 before after` to show exact removed paths. `du` and `uv cache size` describe scale but not artifact identity. These experiments run uv 0.13.3 for cleanup; do not prune the shared cache with uv 0.4.12, which would treat the new buckets as obsolete.

### Normal Prune

#### Verify the Starting Cache

> **Bob:** All five targets should be present. If a bucket is missing, a cleanup result would tell me nothing about that target.

The earlier chapters populated current prebuilt wheels, two current source-built wheels, an unpacked `idna` tree, a centralized environment, and old `wheels-v1`/`built-wheels-v3` entries. Assert that baseline and record it before ordinary prune. `find -P` records symlinks as links, not the archive directories they reference.

The old-version wheel file must actually exist inside `built-wheels-v3`; merely installing into `lab/prune-probe/old-sdist` would not prove it is cached.

In [ ]:
%%bash
set -euxo pipefail
cache=$(uv cache dir)
test -d "$cache/wheels-v7/pypi/numpy"
test -n "$(find "$cache" -type f -name 'uv_cache_demo-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type d -path '*/src/idna' -print)"
test -d "$(<lab/centralized-target)"
test -d "$cache/wheels-v1/pypi/packaging"
test -n "$(find "$cache/built-wheels-v3/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/normal-before.paths
uv cache size --human --preview-features cache-size


#### Prune and Inspect the Results

> **Bob:** `uv cache prune` says ‘unused’. I wonder whether it means ‘not in my `.venv`’ or ‘not reachable or readable in this version's cache format.’

Run ordinary prune and inspect both the things that disappeared and the current-format artifacts that remain. A centralized environment may have a dangling `.venv` link afterward. The old *installed* `idna` and `packaging` copies under `lab/prune-probe/` should still exist. A second prune with no intervening installs should find nothing else to remove.

In [ ]:
%%bash
set -euxo pipefail
uv cache prune
cache=$(uv cache dir)
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/normal-after.paths
test ! -d "$cache/wheels-v1" && test ! -d "$cache/built-wheels-v3"
test ! -d "$(<lab/centralized-target)"
test -d "$cache/wheels-v7/pypi/numpy"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type d -path '*/src/idna' -print)"
find "$cache" -type f \( -name 'uv_cache_demo-*.whl' -o -name 'idna-*.whl' \) -print
test -d lab/prune-probe/old-sdist/idna && test -d lab/prune-probe/old-wheel/packaging
.venv/bin/python -c 'import numpy; print(numpy.__version__)'
uv cache size --human --preview-features cache-size
comm -23 lab/prune-probe/normal-before.paths lab/prune-probe/normal-after.paths | python3 -c 'import sys; paths=sys.stdin.readlines(); print("".join(p for p in paths if p.startswith(("wheels-v1 ", "built-wheels-v3 ", "environments-v2/")) or ("idna-3.10-py3-none-any.whl" in p and "built-wheels-v3" in p))); print(f"{len(paths)} removed paths; full inventory saved")'
uv cache prune


### CI Prune

#### Restore and Verify the Starting Cache

> **Bob:** Ordinary prune just erased uv 0.4.12's old wheel **and** its source-built `idna` wheel. To test CI prune fairly, I have to build those again, not simply look for the paths it already deleted.

Reinstall old `packaging` and old source-built `idna` with uv 0.4.12 into new targets. Recreate the centralized environment whose `.venv` link was left dangling. Reinstall current prebuilt numerical wheels and force a current source build of `idna` into a new target. This makes the second cleanup's starting state independently measurable. An existing `--target` can cause an audit rather than repopulation, so use **new target paths** and `--reinstall` here.

The local demo wheel remained after normal prune; check it again. The source tree and both old-format buckets must also be present before `--ci` is allowed to run.

In [ ]:
%%bash
set -euxo pipefail
cache=$(uv cache dir)
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$cache" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/ci-old-wheel --reinstall --no-deps --only-binary packaging 'packaging==24.2'
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$cache" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/ci-old-sdist --reinstall --no-deps --no-binary idna 'idna==3.10'
uv pip install --python "$(uv python find 3.12)" --target lab/prune-probe/ci-prebuilt --reinstall --no-deps --only-binary :all: numpy scipy matplotlib
uv pip install --python "$(uv python find 3.12)" --target lab/prune-probe/ci-sdist --reinstall --no-deps --no-binary idna 'idna==3.10'
uv --preview-features centralized-project-envs sync --directory lab/centralized --locked --no-install-project --python 3.12
readlink -f lab/centralized/.venv > lab/centralized-target
test -d "$(<lab/centralized-target)"
test -d "$cache/wheels-v1/pypi/packaging" && test -d "$cache/wheels-v7/pypi/numpy"
test -n "$(find "$cache/built-wheels-v3/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type d -path '*/src/idna' -print)"
test -n "$(find "$cache" -type f -name 'uv_cache_demo-*.whl' -print)"
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/ci-before.paths
uv cache size --human --preview-features cache-size


#### Prune and Inspect the Results

> **Bob:** A CI job can download prebuilt wheels again, but rebuilding from source may take longer. Does `--ci` really keep *both* of my current-format built wheels?

CI prune does everything ordinary prune does **plus** remove current-format prebuilt wheels and unpacked source trees. It keeps current source-built `.whl` files and metadata; uv 0.4.12's source-built wheel still disappears because its entire bucket is obsolete. Inspect the path inventory and confirm that `.venv` in the main project still works as a copy-mode install.

In [ ]:
%%bash
set -euxo pipefail
uv cache prune --ci
cache=$(uv cache dir)
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/ci-after.paths
test ! -d "$cache/wheels-v1" && test ! -d "$cache/built-wheels-v3"
test ! -d "$cache/wheels-v7/pypi/numpy"
test ! -d "$(<lab/centralized-target)"
test -z "$(find "$cache/sdists-v10/pypi/idna/3.10" -type d -path '*/src/idna' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache" -type f -name 'uv_cache_demo-*.whl' -print)"
test -d lab/prune-probe/ci-old-sdist/idna && test -d lab/prune-probe/ci-old-wheel/packaging
uv cache size --human --preview-features cache-size
comm -23 lab/prune-probe/ci-before.paths lab/prune-probe/ci-after.paths | python3 -c 'import sys; paths=sys.stdin.readlines(); print("".join(p for p in paths if p.startswith(("wheels-v1 ", "built-wheels-v3 ", "wheels-v7/pypi/numpy ", "environments-v2/")) or "/src d " in p)); print(f"{len(paths)} removed paths; full inventory saved")'
.venv/bin/python -c 'import numpy; from uv_cache_demo.main import cache_message; print(cache_message())'


### Complete Clean

#### Restore and Verify the Starting Cache

> **Bob:** After CI prune, there are almost no prebuilt wheels or unpacked sources left to clean. Before calling `uv cache clean`, I need to put **all five** targets back.

First use the **package-specific** `uv cache clean idna` to invalidate its retained wheel: `--reinstall --no-binary idna` alone would reuse the wheel and never unpack the source again. The local demo wheel belongs to another package and stays cached. Repeat the old-version installs in new targets; restore numerical wheels with `--only-binary :all:` and rebuild `idna` with `--no-binary`. Recreate the centralized environment. Capture a new before-inventory and assert that every target exists. This avoids mistaking CI prune's earlier deletions for the effects of the final, complete `clean`.

In [ ]:
%%bash
set -euxo pipefail
cache=$(uv cache dir)
uv cache clean idna
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$cache" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/clean-old-wheel --reinstall --no-deps --only-binary packaging 'packaging==24.2'
UV_CACHE_DIR="$HOME/.cache/uvx-prune-bootstrap" uvx --from 'uv==0.4.12' uv --cache-dir "$cache" pip install --python "$(uv python find 3.12)" --target lab/prune-probe/clean-old-sdist --reinstall --no-deps --no-binary idna 'idna==3.10'
uv pip install --python "$(uv python find 3.12)" --target lab/prune-probe/clean-prebuilt --reinstall --no-deps --only-binary :all: numpy scipy matplotlib
uv pip install --python "$(uv python find 3.12)" --target lab/prune-probe/clean-sdist --reinstall --no-deps --no-binary idna 'idna==3.10'
uv --preview-features centralized-project-envs sync --directory lab/centralized --locked --no-install-project --python 3.12
readlink -f lab/centralized/.venv > lab/centralized-target
test -d "$(<lab/centralized-target)"
test -d "$cache/wheels-v1/pypi/packaging" && test -d "$cache/wheels-v7/pypi/numpy"
test -n "$(find "$cache/built-wheels-v3/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type f -name 'idna-*.whl' -print)"
test -n "$(find "$cache/sdists-v10/pypi/idna/3.10" -type d -path '*/src/idna' -print)"
test -n "$(find "$cache" -type f -name 'uv_cache_demo-*.whl' -print)"
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/clean-before.paths
uv cache size --human --preview-features cache-size


#### Clean and Inspect the Results

> **Bob:** This should erase *all* reusable package-cache entries, including the locally built wheels. It should not erase the report or packages already copied into `.venv` and my `--target` folders.

Run `uv cache clean` with no package argument. uv 0.13.3 leaves the cache root and its lock file to coordinate processes; do not confuse those with reusable artifacts. Compare before and after inventories, then check the installed copies and report independently.

In [ ]:
%%bash
set -euxo pipefail
uv cache clean
cache=$(uv cache dir)
find -P "$cache" -mindepth 1 -printf '%P %y %s\n' | LC_ALL=C sort > lab/prune-probe/clean-after.paths
test ! -d "$cache/wheels-v1" && test ! -d "$cache/built-wheels-v3"
test ! -d "$cache/wheels-v7" && test ! -d "$cache/sdists-v10"
test ! -d "$(<lab/centralized-target)"
test -z "$(find "$cache" -mindepth 1 ! -name .lock -print)"
uv cache size --human --preview-features cache-size
comm -23 lab/prune-probe/clean-before.paths lab/prune-probe/clean-after.paths | python3 -c 'import sys; paths=sys.stdin.readlines(); print("".join(p for p in paths if p.startswith(("wheels-v1 ", "built-wheels-v3 ", "wheels-v7 ", "sdists-v10 ", "environments-v2/")) or "idna-3.10-py3-none-any.whl" in p or "uv_cache_demo-" in p)); print(f"{len(paths)} removed paths; full inventory saved")'
test -d lab/prune-probe/clean-old-sdist/idna && test -d lab/prune-probe/clean-old-wheel/packaging
.venv/bin/python -c 'import numpy; from uv_cache_demo.main import cache_message; print(cache_message())'
test -f reports/embeddings.png


### Five-Target Comparison

> **Bob:** Now ‘unused’ makes sense: the *cache's references and format* decide what ordinary prune can remove. It never walked my installed environments to ask whether I still use `idna`.

These outcomes describe the verified starting states above and **uv 0.13.3** performing each cleanup. ✓ means removed; — means retained. The old-version row includes uv 0.4.12's *source-built wheel* inside `built-wheels-v3`.

| Cache target | `uv cache prune` | `uv cache prune --ci` | `uv cache clean` |
| --- | :---: | :---: | :---: |
| Current-format prebuilt wheel (`wheels-v7`) | — | ✓ | ✓ |
| Current-format source-built wheel (`sdists-v10/*.whl`) | — | — | ✓ |
| Current unpacked source (`sdists-v10/**/src`) | — | ✓ | ✓ |
| Centralized project environment (`environments-v2`) | ✓ | ✓ | ✓ |
| Obsolete-format entries (`wheels-v1`, `built-wheels-v3`) | ✓ | ✓ | ✓ |

Ordinary prune can also discard unreferenced unpacked-wheel archives and temporary build environments. It does **not** mean ‘remove every wheel not installed in `.venv`.’ To examine every removed path rather than the short printed samples, run `comm -23 lab/prune-probe/normal-before.paths lab/prune-probe/normal-after.paths` and repeat with `ci` or `clean`.

## Epilogue — Bob’s Runbook

> **Bob:** For a regular cleanup, I can prune stale cache formats and centralized environments without throwing away useful current wheels. For a CI runner, `--ci` also drops downloads and unpacked sources but keeps current-format built wheels. When I truly want to start over, `clean` empties the package cache. My copy-mode `.venv` and the report survive all three.

See the companion [cache chapter](../../docs/chapter-03/section-03/uv_cache.md) for the reference commands and [CI cache chapter](../../docs/chapter-03/section-03/uv_cache_ci.md) for the GitHub Actions example.